## Fine-tune Qwen3-Embedding-0.6B with the story-qadataset
This script provides a complete pipeline for fine-tuning the Qwen3-Embedding-0.6B model
on the MSRS story-qa dataset with comprehensive error handling and optimizations.

### References
- [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)
- [MSRS Story-QA](https://huggingface.co/datasets/yale-nlp/MSRS/viewer/story-qa)
- [MSRS Story-Corpus](https://huggingface.co/datasets/yale-nlp/MSRS/viewer/story-corpus)
- https://www.perplexity.ai/search/create-a-comprehensive-step-by-WXkqe_BLSK6qWb7v_DF.PA#0
- https://blog.gopenai.com/fine-tuning-embeddings-for-specific-domains-a-comprehensive-guide-5e4298b42185

### Step 0: Environment Setup

#### 0.1 Install Dependencies

In [1]:
%pip install torch transformers datasets sentence-transformers info-nce-pytorch wandb tqdm mteb optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


#### 0.2 Import Required Libraries

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    AutoModel, AutoTokenizer, AutoConfig,
    TrainingArguments, Trainer,
    get_linear_schedule_with_warmup
)
from transformers.trainer_utils import EvalLoopOutput
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer, losses, evaluation
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
import json
import os
from datetime import datetime
import wandb
import logging
from info_nce import InfoNCE
from tqdm import tqdm

#### 0.3 Configure Logging

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

#### 0.4 Configuration dictionary for all hyperparameters


In [4]:
CONFIG = {
    'model_name': 'Qwen/Qwen3-Embedding-0.6B',
    'max_length': 512,
    'embedding_dim': 1024,
    'temperature': 0.05,
    'split_ratio': 0.7,
    'batch_size': 4,
    'gradient_accumulation_steps': 4,
    'learning_rate': 2e-5,
    'num_epochs': 3,
    'warmup_steps': 500,
    'weight_decay': 0.01,
    'seed': 42,
    'output_dir': './qwen3-embedding-finetuned',
    'use_wandb': True,
    'fp16': torch.cuda.is_available(),
}

#### 0.5 Device Configuration

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


#### 0.6 Set Seed

In [6]:
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG['seed'])

### Step 1: Dataset Loading and Preparation

In [7]:
def load_story_qa_dataset():
    """Load and explore the MSRS story-qa dataset structure."""
    try:
        logger.info("Loading MSRS story-qa dataset...")
        dataset = load_dataset("yale-nlp/MSRS", "story-qa")

        logger.info(f"Dataset splits: {list(dataset.keys())}")
        logger.info(f"Train examples: {len(dataset['train'])}")

        # Examine dataset structure
        sample = dataset['train'][0]
        logger.info(f"Sample keys: {list(sample.keys())}")

        return dataset
    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        raise


def load_story_corpus():
    """Load the story corpus for retrieving full story texts."""
    try:
        logger.info("Loading story corpus...")
        story_dataset = load_dataset("yale-nlp/MSRS", "story-corpus")

        logger.info(f"Corpus size: {len(story_dataset['corpus'])}")

        return story_dataset
    except Exception as e:
        logger.error(f"Error loading story corpus: {e}")
        raise


# Load datasets
qa_dataset = load_story_qa_dataset()
story_dataset = load_story_corpus()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

dev.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/125 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/260 [00:00<?, ? examples/s]

corpus.jsonl: 0.00B [00:00, ?B/s]

Generating corpus split:   0%|          | 0/1138 [00:00<?, ? examples/s]

### Step 2: Data Processing

In [8]:
class StoryQADataProcessor:
    """Process MSRS Story-QA data for contrastive learning with optimizations."""

    def __init__(self, tokenizer, story_corpus, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Create lookup dictionary for efficient story retrieval
        logger.info("Creating story lookup dictionary...")
        self.story_lookup = {}
        for item in tqdm(story_corpus['corpus'], desc="Building story index"):
            try:
                story_id = item.get('id')
                story_text = item.get('text', '')
                if story_id and story_text:
                    self.story_lookup[story_id] = story_text
            except Exception as e:
                logger.warning(f"Error processing corpus item: {e}")
                continue

        logger.info(f"Story lookup created with {len(self.story_lookup)} entries")

    def create_contrastive_pairs(self, examples):
        """
        Create (query, positive_passage) pairs from story-qa data.
        Each query should be paired with relevant story passages.
        """
        if not examples:
            raise ValueError("Empty examples list provided")

        queries = []
        positives = []
        skipped = 0

        logger.info(f"Processing {len(examples)} examples...")

        for idx, example in enumerate(tqdm(examples, desc="Creating contrastive pairs")):
            try:
                # Extract query and relevant passages
                query = example.get('query', '').strip()
                if not query:
                    skipped += 1
                    continue

                positive_passages = example.get('gold_documents', [])
                if not positive_passages:
                    skipped += 1
                    continue

                # Create pairs for each positive passage
                for story_id in positive_passages:
                    story_text = self.story_lookup.get(story_id)

                    if not story_text:
                        logger.debug(f"Story ID {story_id} not found in corpus")
                        continue

                    queries.append(query)
                    positives.append(story_text)

            except Exception as e:
                logger.warning(f"Error processing example {idx}: {e}")
                skipped += 1
                continue

        logger.info(f"Created {len(queries)} query-positive pairs (skipped {skipped} invalid examples)")

        if len(queries) == 0:
            raise ValueError("No valid query-positive pairs created from dataset")

        return {
            'query': queries,
            'positive': positives
        }

### Step 3: Model Architecture

In [9]:
class Qwen3EmbeddingModel(nn.Module):
    """Custom Qwen3 embedding model for contrastive learning with optimizations."""

    def __init__(self, model_name="Qwen/Qwen3-Embedding-0.6B",
                 embedding_dim=1024, temperature=0.05):
        super().__init__()

        logger.info(f"Initializing {model_name}...")

        # Load base model
        self.config = AutoConfig.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.embedding_dim = embedding_dim
        self.temperature = temperature

        # Initialize InfoNCE loss
        self.info_nce_loss = InfoNCE(temperature=temperature)

        # Projection layer (if needed)
        if self.model.config.hidden_size != embedding_dim:
            self.projection = nn.Linear(
                self.model.config.hidden_size,
                embedding_dim
            )
            logger.info(f"Added projection layer: {self.model.config.hidden_size} -> {embedding_dim}")
        else:
            self.projection = nn.Identity()

        # Layer normalization
        self.layer_norm = nn.LayerNorm(embedding_dim)

        logger.info(f"Model initialized with {sum(p.numel() for p in self.parameters()):,} parameters")

    def mean_pooling(self, token_embeddings, attention_mask):
        """Apply mean pooling to get sentence embeddings."""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
            input_mask_expanded.sum(1), min=1e-9
        )

    def encode(self, input_ids, attention_mask):
        """Encode text to embeddings."""
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        # Use mean pooling
        if hasattr(outputs, 'last_hidden_state'):
            embeddings = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        else:
            embeddings = outputs.pooler_output

        # Apply projection and normalization
        embeddings = self.projection(embeddings)
        embeddings = self.layer_norm(embeddings)

        # L2 normalize for cosine similarity
        embeddings = nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings

    def forward(self, query_input_ids, query_attention_mask,
                positive_input_ids, positive_attention_mask):
        """Forward pass with InfoNCE loss."""

        # Encode queries and positives
        query_embeddings = self.encode(query_input_ids, query_attention_mask)
        positive_embeddings = self.encode(positive_input_ids, positive_attention_mask)

        # Compute InfoNCE loss with in-batch negatives
        loss = self.info_nce_loss(query_embeddings, positive_embeddings)

        return {
            'loss': loss,
            'query_embeddings': query_embeddings,
            'positive_embeddings': positive_embeddings
        }

    def gradient_checkpointing_enable(self, **kwargs):
        """Enable gradient checkpointing for the model."""
        if hasattr(self.model, 'gradient_checkpointing_enable'):
            self.model.gradient_checkpointing_enable(**kwargs)
            logger.info("Gradient checkpointing enabled")
        else:
            logger.warning("Base model does not support gradient checkpointing")

### Step 4: Data Collator

In [10]:
class ContrastiveDataCollator:
    """Data collator for contrastive learning with in-batch negatives."""

    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        """Collate batch for contrastive training."""
        if not batch:
            raise ValueError("Empty batch provided to collator")

        queries = [item['query'] for item in batch]
        positives = [item['positive'] for item in batch]

        # Tokenize queries
        query_tokens = self.tokenizer(
            queries,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Tokenize positives
        positive_tokens = self.tokenizer(
            positives,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'query_input_ids': query_tokens['input_ids'],
            'query_attention_mask': query_tokens['attention_mask'],
            'positive_input_ids': positive_tokens['input_ids'],
            'positive_attention_mask': positive_tokens['attention_mask']
        }

### Step 5: Custom Trainer

In [11]:
class ContrastiveTrainer(Trainer):
    """Custom trainer for contrastive learning with proper evaluation."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.best_metric = float('inf')

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """Compute contrastive loss."""
        outputs = model(**inputs)
        loss = outputs['loss']

        if return_outputs:
            return (loss, outputs)
        return loss

    def evaluation_loop(self, dataloader, description, prediction_loss_only=None,
                       ignore_keys=None, metric_key_prefix="eval"):
        """Custom evaluation loop with proper return type."""
        model = self.model
        model.eval()

        total_loss = 0
        num_samples = 0

        logger.info(f"Starting evaluation: {description}")

        with torch.no_grad():
            for batch in tqdm(dataloader, desc=description):
                # Move batch to device
                batch = {k: v.to(self.args.device) for k, v in batch.items()}

                # Forward pass
                outputs = model(**batch)
                loss = outputs['loss']

                total_loss += loss.item()
                num_samples += 1

        avg_loss = total_loss / num_samples if num_samples > 0 else 0

        metrics = {f"{metric_key_prefix}_loss": avg_loss}

        # Track best model
        if avg_loss < self.best_metric:
            self.best_metric = avg_loss
            logger.info(f"New best model! Validation loss: {avg_loss:.4f}")

        # Return proper EvalLoopOutput
        return EvalLoopOutput(
            predictions=None,
            label_ids=None,
            metrics=metrics,
            num_samples=num_samples
        )


Step 6: Evaluation Utilities

In [12]:
class EmbeddingEvaluator:
    """Comprehensive evaluation for embedding models."""

    def __init__(self, model, tokenizer, device='cuda'):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def encode_texts(self, texts, batch_size=32):
        """Encode a list of texts to embeddings with memory optimization."""
        self.model.eval()
        embeddings = []

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc="Encoding texts"):
                batch_texts = texts[i:i + batch_size]

                # Tokenize
                tokens = self.tokenizer(
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=CONFIG['max_length'],
                    return_tensors='pt'
                ).to(self.device)

                # Encode
                batch_embeddings = self.model.encode(
                    tokens['input_ids'],
                    tokens['attention_mask']
                )

                embeddings.append(batch_embeddings.cpu())

                # Clear CUDA cache periodically
                if i % 10 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()

        return torch.cat(embeddings, dim=0)

    def compute_similarity_metrics(self, queries, passages, labels):
        """Compute retrieval metrics."""
        logger.info("Computing similarity metrics...")

        # Encode queries and passages
        query_embeddings = self.encode_texts(queries)
        passage_embeddings = self.encode_texts(passages)

        # Compute similarities
        similarities = torch.cosine_similarity(
            query_embeddings.unsqueeze(1),
            passage_embeddings.unsqueeze(0),
            dim=2
        )

        # Compute metrics
        metrics = self._compute_retrieval_metrics(similarities, labels)
        return metrics

    def _compute_retrieval_metrics(self, similarities, labels, k_values=[1, 5, 10]):
        """Compute Recall@K, nDCG@K, and MRR."""
        metrics = {}

        # Get top-k predictions
        _, top_indices = similarities.topk(max(k_values), dim=1)

        for k in k_values:
            # Recall@K
            recall_k = self._recall_at_k(top_indices[:, :k], labels, k)
            metrics[f'recall@{k}'] = recall_k

            # nDCG@K
            ndcg_k = self._ndcg_at_k(similarities, labels, k)
            metrics[f'ndcg@{k}'] = ndcg_k

        # MRR
        mrr = self._mean_reciprocal_rank(top_indices, labels)
        metrics['mrr'] = mrr

        return metrics

    def _recall_at_k(self, top_k_indices, labels, k):
        """Calculate Recall@K."""
        correct = 0
        total = len(labels)

        for i, relevant_indices in enumerate(labels):
            if any(idx.item() in relevant_indices for idx in top_k_indices[i]):
                correct += 1

        return correct / total if total > 0 else 0

    def _ndcg_at_k(self, similarities, labels, k):
        """Calculate nDCG@K."""
        ndcg_scores = []

        for i, relevant_indices in enumerate(labels):
            # Get top-k similarities
            top_k_sims, top_k_indices = similarities[i].topk(k)

            # Calculate DCG
            dcg = 0
            for j, idx in enumerate(top_k_indices):
                if idx.item() in relevant_indices:
                    dcg += 1 / np.log2(j + 2)

            # Calculate IDCG
            idcg = sum(1 / np.log2(j + 2) for j in range(min(len(relevant_indices), k)))

            # Calculate nDCG
            ndcg = dcg / idcg if idcg > 0 else 0
            ndcg_scores.append(ndcg)

        return np.mean(ndcg_scores) if ndcg_scores else 0

    def _mean_reciprocal_rank(self, top_indices, labels):
        """Calculate Mean Reciprocal Rank."""
        rr_scores = []

        for i, relevant_indices in enumerate(labels):
            rr = 0
            for j, idx in enumerate(top_indices[i]):
                if idx.item() in relevant_indices:
                    rr = 1 / (j + 1)
                    break
            rr_scores.append(rr)

        return np.mean(rr_scores) if rr_scores else 0


### Step 7: Training Setup

In [13]:
def optimize_batch_size():
    """Automatically find optimal batch size based on GPU memory."""
    if not torch.cuda.is_available():
        return 4

    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9

    if gpu_memory >= 40:  # A100 40GB
        return 32
    elif gpu_memory >= 24:  # RTX 4090 24GB
        return 16
    elif gpu_memory >= 16:  # RTX 4080 16GB
        return 8
    else:
        return 4


def setup_training_args(output_dir, batch_size=None):
    """Configure training arguments for optimal performance."""

    if batch_size is None:
        batch_size = optimize_batch_size()

    logger.info(f"Using batch size: {batch_size}")

    training_args = TrainingArguments(
        output_dir=output_dir,

        # Training schedule
        num_train_epochs=CONFIG['num_epochs'],
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],

        # Learning rates
        learning_rate=CONFIG['learning_rate'],
        warmup_steps=CONFIG['warmup_steps'],
        lr_scheduler_type="cosine",

        # Optimization
        weight_decay=CONFIG['weight_decay'],
        adam_beta1=0.9,
        adam_beta2=0.999,
        max_grad_norm=1.0,

        # Mixed precision
        fp16=CONFIG['fp16'],
        dataloader_pin_memory=True,

        # Evaluation and saving
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # Logging
        logging_dir=f"{output_dir}/logs",
        logging_steps=100,
        report_to="wandb" if CONFIG['use_wandb'] else "none",

        # Reproducibility
        seed=CONFIG['seed'],
        data_seed=CONFIG['seed'],

        # Performance
        remove_unused_columns=False,
        label_smoothing_factor=0.1,
        gradient_checkpointing=True,

        # DataLoader
        dataloader_num_workers=4,
    )

    return training_args


def prepare_training_data(dataset, processor, split_ratio=0.7):
    """Prepare training and validation datasets with shuffling."""

    logger.info("Preparing training data...")

    # Process the dataset
    train_data = dataset['train']

    # Create contrastive pairs
    processed_data = processor.create_contrastive_pairs(train_data)

    # Convert to Dataset object
    contrastive_dataset = Dataset.from_dict(processed_data)

    # Shuffle before splitting for better train/val distribution
    contrastive_dataset = contrastive_dataset.shuffle(seed=CONFIG['seed'])

    # Split into train/validation
    train_size = int(len(contrastive_dataset) * split_ratio)
    train_dataset = contrastive_dataset.select(range(train_size))
    val_dataset = contrastive_dataset.select(range(train_size, len(contrastive_dataset)))

    logger.info(f"Training examples: {len(train_dataset)}")
    logger.info(f"Validation examples: {len(val_dataset)}")

    return train_dataset, val_dataset


### Step 8: Main Training Loop

In [14]:
def main_training_loop():
    """Main training execution with comprehensive error handling."""

    wandb_initialized = False

    try:
        # Initialize components
        logger.info("Initializing training components...")

        # Tokenizer
        tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

        # Data processor
        processor = StoryQADataProcessor(tokenizer, story_dataset, CONFIG['max_length'])

        # Prepare datasets
        train_dataset, val_dataset = prepare_training_data(
            qa_dataset, processor, CONFIG['split_ratio']
        )

        # Initialize model and move to device
        model = Qwen3EmbeddingModel(
            model_name=CONFIG['model_name'],
            embedding_dim=CONFIG['embedding_dim'],
            temperature=CONFIG['temperature']
        ).to(device)

        # Data collator
        data_collator = ContrastiveDataCollator(tokenizer, CONFIG['max_length'])

        # Training arguments
        training_args = setup_training_args(
            output_dir=CONFIG['output_dir'],
            batch_size=CONFIG['batch_size']
        )

        # Initialize wandb
        if CONFIG['use_wandb']:
            wandb.init(
                project="qwen3-embedding-finetune",
                name=f"story-qa-{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                config=CONFIG
            )
            wandb_initialized = True

        # Initialize trainer
        trainer = ContrastiveTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            tokenizer=tokenizer,
        )

        # Validation before training
        logger.info("Running pre-training validation...")
        pre_train_metrics = trainer.evaluate()
        logger.info(f"Pre-training metrics: {pre_train_metrics}")

        # Start training
        logger.info("=" * 50)
        logger.info("Starting training...")
        logger.info("=" * 50)

        train_result = trainer.train()

        logger.info("=" * 50)
        logger.info("Training completed!")
        logger.info("=" * 50)

        # Save final model
        output_path = training_args.output_dir
        os.makedirs(output_path, exist_ok=True)

        # Save model state dict
        torch.save(
            model.state_dict(),
            os.path.join(output_path, "pytorch_model.bin")
        )
        tokenizer.save_pretrained(output_path)

        # Save training config
        with open(os.path.join(output_path, "training_config.json"), 'w') as f:
            json.dump(CONFIG, f, indent=2)

        logger.info(f"Model saved to {output_path}")

        return trainer, model, tokenizer

    except Exception as e:
        logger.error(f"Training failed with error: {e}", exc_info=True)
        raise
    finally:
        if wandb_initialized:
            wandb.finish()

Step 9: Comprehensive Evaluation

In [15]:
def comprehensive_evaluation(trainer, model, tokenizer, val_dataset):
    """Run comprehensive evaluation on the fine-tuned model."""

    logger.info("=" * 50)
    logger.info("Starting comprehensive evaluation...")
    logger.info("=" * 50)

    wandb_initialized = False

    try:
        # Load best model checkpoint
        if trainer.state.best_model_checkpoint:
            logger.info(f"Loading best model from {trainer.state.best_model_checkpoint}")
            model.load_state_dict(
                torch.load(
                    f"{trainer.state.best_model_checkpoint}/pytorch_model.bin",
                    map_location=device,
                    weights_only=True
                )
            )
        else:
            logger.warning("No best model checkpoint found, using current model state")

        model.eval()

        # Temporarily disable wandb logging for evaluation
        #original_report_to = trainer.args.report_to
        #trainer.args.report_to = []

        # Initialize wandb for evaluation if not already
        if CONFIG['use_wandb'] and not wandb.run:
             wandb.init(
                project="qwen3-embedding-finetune",
                name=f"story-qa-evaluation-{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                config=CONFIG,
                reinit=True # Allow re-initialization
            )
             wandb_initialized = True


        # Evaluate on validation set
        logger.info("Evaluating on validation set...")
        val_metrics = trainer.evaluate()

        # Restore original reporting settings
        #trainer.args.report_to = original_report_to

        logger.info(f"Validation metrics: {json.dumps(val_metrics, indent=2)}")

        # Custom retrieval evaluation
        retrieval_metrics = None
        if len(val_dataset) > 0:
            logger.info("Running retrieval evaluation...")

            # Create evaluator
            evaluator = EmbeddingEvaluator(model, tokenizer, device)

            # Prepare evaluation data
            sample_size = min(100, len(val_dataset))
            sample_queries = [val_dataset[i]['query'] for i in range(sample_size)]
            sample_passages = [val_dataset[i]['positive'] for i in range(sample_size)]
            sample_labels = [[i] for i in range(sample_size)]

            retrieval_metrics = evaluator.compute_similarity_metrics(
                sample_queries, sample_passages, sample_labels
            )
            logger.info(f"Retrieval metrics: {json.dumps(retrieval_metrics, indent=2)}")

        # Save evaluation results
        results = {
            'validation': val_metrics,
            'retrieval': retrieval_metrics,
            'timestamp': datetime.now().isoformat()
        }

        results_path = os.path.join(CONFIG['output_dir'], 'evaluation_results.json')
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)

        logger.info(f"Evaluation results saved to {results_path}")

        return results

    except Exception as e:
        logger.error(f"Evaluation failed with error: {e}", exc_info=True)
        return None
    finally:
        if wandb_initialized:
            wandb.finish()

### Step 10: Model Saving and Inference

In [16]:
def save_model_for_inference(model, tokenizer, model_path="./qwen3-embedding-final"):
    """Save model for easy inference."""

    logger.info(f"Saving model for inference to {model_path}...")

    try:
        os.makedirs(model_path, exist_ok=True)

        # Save model state dict
        torch.save(model.state_dict(), f"{model_path}/pytorch_model.bin")

        # Save tokenizer
        tokenizer.save_pretrained(model_path)

        # Save config
        config = {
            "model_type": "qwen3-embedding",
            "base_model": CONFIG['model_name'],
            "embedding_dim": CONFIG['embedding_dim'],
            "temperature": CONFIG['temperature'],
            "fine_tuned_on": "MSRS-story-qa",
            "training_date": datetime.now().isoformat(),
            "max_length": CONFIG['max_length']
        }

        with open(f"{model_path}/config.json", 'w') as f:
            json.dump(config, f, indent=2)

        logger.info(f"Model successfully saved to {model_path}")

    except Exception as e:
        logger.error(f"Failed to save model: {e}", exc_info=True)
        raise


def inference_example(model, tokenizer):
    """Demonstrate model usage for inference."""

    logger.info("Running inference example...")

    model.eval()

    # Example queries and passages
    queries = [
        "What happened to the main character in the story?",
        "Who was the antagonist in the tale?",
        "How did the story end?"
    ]

    passages = [
        "The brave knight defeated the dragon and saved the kingdom.",
        "The evil wizard cast a spell on the village.",
        "Everyone lived happily ever after in the enchanted forest."
    ]

    # Create evaluator
    evaluator = EmbeddingEvaluator(model, tokenizer, device)

    # Encode texts
    query_embeddings = evaluator.encode_texts(queries)
    passage_embeddings = evaluator.encode_texts(passages)

    # Compute similarities
    similarities = torch.cosine_similarity(
        query_embeddings.unsqueeze(1),
        passage_embeddings.unsqueeze(0),
        dim=2
    )

    # Display results
    logger.info("\n" + "=" * 70)
    logger.info("Query-Passage Similarities:")
    logger.info("=" * 70)

    for i, query in enumerate(queries):
        logger.info(f"\nQuery {i+1}: {query}")
        for j, passage in enumerate(passages):
            sim_score = similarities[i, j].item()
            logger.info(f"  Passage {j+1} (similarity: {sim_score:.4f}):")
            logger.info(f"    {passage}")

    logger.info("=" * 70)

### Step 11: Main Execution

In [17]:
def main():
    """Main execution function."""

    logger.info("=" * 70)
    logger.info("QWEN3-EMBEDDING-0.6B FINE-TUNING PIPELINE")
    logger.info("=" * 70)
    logger.info(f"Configuration: {json.dumps(CONFIG, indent=2)}")
    logger.info("=" * 70)

    try:
        # Step 1: Train the model
        trainer, model, tokenizer = main_training_loop()

        # Step 2: Evaluate the model
        evaluation_results = comprehensive_evaluation(
            trainer, model, tokenizer,
            trainer.eval_dataset
        )

        # Step 3: Save model for inference
        save_model_for_inference(model, tokenizer)

        # Step 4: Run inference example
        inference_example(model, tokenizer)

        logger.info("=" * 70)
        logger.info("Pipeline completed successfully!")
        logger.info("=" * 70)

        return {
            'trainer': trainer,
            'model': model,
            'tokenizer': tokenizer,
            'evaluation_results': evaluation_results
        }

    except Exception as e:
        logger.error(f"Pipeline failed: {e}", exc_info=True)
        raise



### Step 12: Optional Advanced Features

In [18]:
def hyperparameter_search(n_trials=10):
    """Perform hyperparameter optimization using Optuna."""
    try:
        import optuna
    except ImportError:
        logger.error("Optuna not installed. Install with: pip install optuna")
        return None

    logger.info(f"Starting hyperparameter search with {n_trials} trials...")

    def objective(trial):
        # Suggest hyperparameters
        learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)
        temperature = trial.suggest_uniform('temperature', 0.01, 0.2)
        batch_size = trial.suggest_categorical('batch_size', [4, 8, 16])
        weight_decay = trial.suggest_loguniform('weight_decay', 1e-4, 1e-1)

        # Update config
        trial_config = CONFIG.copy()
        trial_config['learning_rate'] = learning_rate
        trial_config['temperature'] = temperature
        trial_config['batch_size'] = batch_size
        trial_config['weight_decay'] = weight_decay
        trial_config['output_dir'] = f"./qwen3-trial-{trial.number}"

        try:
            # Initialize components
            tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
            processor = StoryQADataProcessor(tokenizer, story_dataset, CONFIG['max_length'])
            train_dataset, val_dataset = prepare_training_data(qa_dataset, processor, CONFIG['split_ratio'])

            # Initialize model
            model = Qwen3EmbeddingModel(
                model_name=CONFIG['model_name'],
                embedding_dim=CONFIG['embedding_dim'],
                temperature=temperature
            ).to(device)

            # Data collator
            data_collator = ContrastiveDataCollator(tokenizer, CONFIG['max_length'])

            # Training arguments
            training_args = setup_training_args(
                output_dir=trial_config['output_dir'],
                batch_size=batch_size
            )
            training_args.learning_rate = learning_rate
            training_args.weight_decay = weight_decay
            training_args.num_train_epochs = 1  # Reduce epochs for HPO

            # Train
            trainer = ContrastiveTrainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
                data_collator=data_collator,
                tokenizer=tokenizer,
            )

            trainer.train()
            eval_results = trainer.evaluate()

            return eval_results['eval_loss']

        except Exception as e:
            logger.error(f"Trial {trial.number} failed: {e}")
            return float('inf')

    # Run optimization
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    logger.info(f"Best parameters: {study.best_params}")
    logger.info(f"Best validation loss: {study.best_value}")

    return study.best_params


def evaluate_on_mteb_subset():
    """Evaluate model on MTEB retrieval tasks (requires mteb library)."""
    try:
        from mteb import MTEB
    except ImportError:
        logger.warning("MTEB not installed. Install with: pip install mteb")
        return None

    logger.info("Running MTEB evaluation...")

    # Load fine-tuned model
    model_path = CONFIG['output_dir']

    # Initialize MTEB evaluation
    evaluation = MTEB(tasks=["QuoraRetrieval", "NFCorpus", "SciFact"])

    # Convert model to MTEB-compatible format
    class MTEBModel:
        def __init__(self, model_path):
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = Qwen3EmbeddingModel().to(device)
            self.model.load_state_dict(
                torch.load(f"{model_path}/pytorch_model.bin", map_location=device)
            )
            self.model.eval()

        def encode(self, sentences, batch_size=32, **kwargs):
            evaluator = EmbeddingEvaluator(self.model, self.tokenizer, device)
            embeddings = evaluator.encode_texts(sentences, batch_size)
            return embeddings.numpy()

    mteb_model = MTEBModel(model_path)

    # Run evaluation
    results = evaluation.run(mteb_model, output_folder=f"{model_path}/mteb_results")

    logger.info(f"MTEB results: {results}")
    return results


def export_to_onnx(model, tokenizer, output_path="./model.onnx"):
    """Export model to ONNX format for production deployment."""
    try:
        import torch.onnx
    except ImportError:
        logger.error("ONNX export requires torch.onnx")
        return False

    logger.info(f"Exporting model to ONNX format: {output_path}")

    model.eval()

    # Create dummy input
    dummy_text = ["This is a sample text for ONNX export"]
    dummy_tokens = tokenizer(
        dummy_text,
        padding=True,
        truncation=True,
        max_length=CONFIG['max_length'],
        return_tensors='pt'
    ).to(device)

    # Export
    torch.onnx.export(
        model,
        (dummy_tokens['input_ids'], dummy_tokens['attention_mask']),
        output_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input_ids', 'attention_mask'],
        output_names=['embeddings'],
        dynamic_axes={
            'input_ids': {0: 'batch_size', 1: 'sequence'},
            'attention_mask': {0: 'batch_size', 1: 'sequence'},
            'embeddings': {0: 'batch_size'}
        }
    )

    logger.info(f"Model exported to {output_path}")
    return True

### Step 13: Utilities and Helpers

In [19]:
def load_finetuned_model(model_path, device='cuda'):
    """Load a fine-tuned model for inference."""
    logger.info(f"Loading fine-tuned model from {model_path}")

    # Load config
    with open(f"{model_path}/config.json", 'r') as f:
        config = json.load(f)

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Load model
    model = Qwen3EmbeddingModel(
        model_name=config.get('base_model', CONFIG['model_name']),
        embedding_dim=config.get('embedding_dim', CONFIG['embedding_dim']),
        temperature=config.get('temperature', CONFIG['temperature'])
    )

    # Load weights
    model.load_state_dict(
        torch.load(f"{model_path}/pytorch_model.bin", map_location=device)
    )

    model = model.to(device)
    model.eval()

    logger.info("Model loaded successfully")
    return model, tokenizer


def compare_models(original_model_name, finetuned_model_path, test_queries):
    """Compare original and fine-tuned models."""
    logger.info("Comparing original and fine-tuned models...")

    # Load original model
    logger.info("Loading original model...")
    original_tokenizer = AutoTokenizer.from_pretrained(original_model_name)
    original_model = Qwen3EmbeddingModel(model_name=original_model_name).to(device)
    original_model.eval()

    # Load fine-tuned model
    logger.info("Loading fine-tuned model...")
    finetuned_model, finetuned_tokenizer = load_finetuned_model(finetuned_model_path, device)

    # Create evaluators
    original_evaluator = EmbeddingEvaluator(original_model, original_tokenizer, device)
    finetuned_evaluator = EmbeddingEvaluator(finetuned_model, finetuned_tokenizer, device)

    # Encode test queries
    logger.info("Encoding queries with both models...")
    original_embeddings = original_evaluator.encode_texts(test_queries)
    finetuned_embeddings = finetuned_evaluator.encode_texts(test_queries)

    # Compute embedding differences
    cosine_sims = torch.cosine_similarity(original_embeddings, finetuned_embeddings, dim=1)

    logger.info("\n" + "=" * 70)
    logger.info("Model Comparison Results:")
    logger.info("=" * 70)
    logger.info(f"Average cosine similarity: {cosine_sims.mean().item():.4f}")
    logger.info(f"Min cosine similarity: {cosine_sims.min().item():.4f}")
    logger.info(f"Max cosine similarity: {cosine_sims.max().item():.4f}")

    for i, query in enumerate(test_queries):
        logger.info(f"\nQuery {i+1}: {query}")
        logger.info(f"  Similarity: {cosine_sims[i].item():.4f}")

    logger.info("=" * 70)

    return {
        'average_similarity': cosine_sims.mean().item(),
        'min_similarity': cosine_sims.min().item(),
        'max_similarity': cosine_sims.max().item(),
        'query_similarities': cosine_sims.tolist()
    }


def create_training_report(trainer, evaluation_results, output_path="./training_report.md"):
    """Generate a comprehensive training report."""
    logger.info(f"Creating training report: {output_path}")

    report = f"""# Qwen3-Embedding-0.6B Fine-tuning Report

## Training Configuration

```json
{json.dumps(CONFIG, indent=2)}
```

## Training Results

### Final Metrics

"""

    if evaluation_results and 'validation' in evaluation_results:
        report += f"""
**Validation Loss:** {evaluation_results['validation'].get('eval_loss', 'N/A')}

"""

    if evaluation_results and 'retrieval' in evaluation_results and evaluation_results['retrieval']:
        report += """### Retrieval Metrics

"""
        for metric, value in evaluation_results['retrieval'].items():
            report += f"- **{metric}:** {value:.4f}\n"

    report += f"""

## Training Details

- **Model:** {CONFIG['model_name']}
- **Dataset:** MSRS Story-QA
- **Training Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Device:** {device}
- **GPU:** {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}

## Model Architecture

- **Embedding Dimension:** {CONFIG['embedding_dim']}
- **Temperature:** {CONFIG['temperature']}
- **Max Sequence Length:** {CONFIG['max_length']}

## Training Hyperparameters

- **Batch Size:** {CONFIG['batch_size']}
- **Gradient Accumulation Steps:** {CONFIG['gradient_accumulation_steps']}
- **Learning Rate:** {CONFIG['learning_rate']}
- **Weight Decay:** {CONFIG['weight_decay']}
- **Epochs:** {CONFIG['num_epochs']}
- **Warmup Steps:** {CONFIG['warmup_steps']}

## Training History

"""

    # Add training history if available
    if hasattr(trainer, 'state') and trainer.state.log_history:
        report += "| Step | Loss | Learning Rate |\n"
        report += "|------|------|---------------|\n"

        for log_entry in trainer.state.log_history[-10:]:  # Last 10 entries
            step = log_entry.get('step', 'N/A')
            loss = log_entry.get('loss', 'N/A')
            lr = log_entry.get('learning_rate', 'N/A')

            if loss != 'N/A':
                loss = f"{loss:.4f}"
            if lr != 'N/A':
                lr = f"{lr:.2e}"

            report += f"| {step} | {loss} | {lr} |\n"

    report += f"""

## Output Files

- **Model Checkpoint:** `{CONFIG['output_dir']}`
- **Training Logs:** `{CONFIG['output_dir']}/logs`
- **Evaluation Results:** `{CONFIG['output_dir']}/evaluation_results.json`

## Usage

```python
from transformers import AutoTokenizer
from your_module import Qwen3EmbeddingModel, load_finetuned_model

# Load the fine-tuned model
model, tokenizer = load_finetuned_model("{CONFIG['output_dir']}")

# Encode text
texts = ["Your text here"]
tokens = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
embeddings = model.encode(tokens['input_ids'], tokens['attention_mask'])
```

---
*Report generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*
"""

    # Save report
    with open(output_path, 'w') as f:
        f.write(report)

    logger.info(f"Training report saved to {output_path}")


## Main Execution

In [20]:
# Run the main pipeline
results = main()

# Create training report
if results and results.get('trainer') and results.get('evaluation_results'):
    create_training_report(
        results['trainer'],
        results['evaluation_results'],
        output_path=f"{CONFIG['output_dir']}/training_report.md"
    )

logger.info("All tasks completed!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Creating contrastive pairs: 100%|██████████| 250/250 [00:00<00:00, 3626.89it/s]


...

eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▄██
train/global_step,▁▃▆██
train/grad_norm,█▄▁
train/learning_rate,▁▅█
train/loss,█▁▁
eval/loss,1.04142
eval/runtime,131.8364


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Encoding texts: 100%|██████████| 4/4 [00:16<00:00,  4.16s/it]


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁
train/global_step,▁
eval/loss,0.12886
eval/runtime,63.3389
eval/samples_per_second,2.731
eval/steps_per_second,0.695
train/epoch,3


Encoding texts: 100%|██████████| 1/1 [00:00<00:00, 18.36it/s]


In [23]:
# Example usage of compare_models
# Define some test queries
test_queries = [
    "What is the main theme of the story?",
    "Describe the personality of the protagonist.",
    "What was the conflict in the narrative?",
    "Summarize the key events in the plot.",
    "How did the setting influence the story?"
]

# Define the path to your fine-tuned model
finetuned_model_path = 'qwen3-embedding-final'

# Compare the models
comparison_results = compare_models(
    original_model_name=CONFIG['model_name'],
    finetuned_model_path=finetuned_model_path,
    test_queries=test_queries
)

# Display the comparison results
print("\nComparison Results:")
print(json.dumps(comparison_results, indent=2))

Encoding texts: 100%|██████████| 1/1 [00:00<00:00, 11.33it/s]



Comparison Results:
{
  "average_similarity": 0.7386413812637329,
  "min_similarity": 0.6902546882629395,
  "max_similarity": 0.7779222726821899,
  "query_similarities": [
    0.7779222726821899,
    0.7511777877807617,
    0.7569202184677124,
    0.7169317007064819,
    0.6902546882629395
  ]
}


In [26]:
# Optional: Run MTEB evaluation (uncomment to use)
mteb_results = evaluate_on_mteb_subset()
mteb_results

/usr/local/lib/python3.12/dist-packages/mteb/evaluation/MTEB.py:120: UserWarning: Passing task names as strings is deprecated and will be removed in 2.0 release. Please use `tasks = mteb.get_tasks(tasks=[...])` method to get tasks instead.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 594.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 456.12 MiB is free. Process 25094 has 14.29 GiB memory in use. Of the allocated memory 13.65 GiB is allocated by PyTorch, and 524.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [25]:
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")
else:
    print("CUDA not available, no cache to clear.")

CUDA cache cleared.


In [27]:
# Optional: Run hyperparameter search (uncomment to use)
best_params = hyperparameter_search(n_trials=10)
best_params

[I 2025-10-14 09:35:54,510] A new study created in memory with name: no-name-56c1bf90-80c5-44bd-98d6-4437a20a6334
/tmp/ipython-input-1318563015.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)
/tmp/ipython-input-1318563015.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  temperature = trial.suggest_uniform('temperature', 0.01, 0.2)
/tmp/ipython-input-1318563015.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform(

{'learning_rate': 3.5527268483002074e-06,
 'temperature': 0.027531569369334014,
 'batch_size': 16,
 'weight_decay': 0.00511654494951919}

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Optional: Export to ONNX (uncomment to use)
# if results and results.get('model') and results.get('tokenizer'):
#     export_to_onnx(
#         results['model'],
#         results['tokenizer'],
#         output_path=f"{CONFIG['output_dir']}/model.onnx"
#     )